# Phase 0: Machine & Environment Audit

## Objective
Inspect the local execution environment to determine hardware constraints, CUDA availability, GPU VRAM, RAM, and disk space. This audit guides decisions regarding local embedding/reranking inference vs. API-based models (Gemini, Sarvam, ElevenLabs).

In [1]:
import sys
import platform
import os
import psutil
import shutil

print("=== SYSTEM & HARDWARE AUDIT ===")
print(f"Python Version: {sys.version}")
print(f"Platform OS: {platform.platform()}")
print(f"Architecture: {platform.architecture()[0]}")
print(f"Logical CPU Cores: {os.cpu_count()}")
print(f"Total RAM: {psutil.virtual_memory().total / (1024**3):.2f} GB")
print(f"Available RAM: {psutil.virtual_memory().available / (1024**3):.2f} GB")
print(f"Free Disk Space: {shutil.disk_usage('.').free / (1024**3):.2f} GB")

=== SYSTEM & HARDWARE AUDIT ===
Python Version: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]
Platform OS: Windows-11-10.0.26200-SP0
Architecture: 64bit
Logical CPU Cores: 16
Total RAM: 15.82 GB
Available RAM: 3.11 GB
Free Disk Space: 1.04 GB


In [2]:
try:
    import torch
    print("=== PYTORCH & CUDA ACCELERATION ===")
    print(f"PyTorch Version: {torch.__version__}")
    cuda_avail = torch.cuda.is_available()
    print(f"CUDA Available: {cuda_avail}")
    if cuda_avail:
        print(f"Device Count: {torch.cuda.device_count()}")
        print(f"Device Name: {torch.cuda.get_device_name(0)}")
        print(f"VRAM Total: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    else:
        print("CUDA acceleration unavailable. System will use CPU fallback with CPU-optimized vector indexes (FAISS CPU, BM25).")
except ImportError:
    print("PyTorch is not installed in the current environment.")

=== PYTORCH & CUDA ACCELERATION ===
PyTorch Version: 2.13.0+cpu
CUDA Available: False
CUDA acceleration unavailable. System will use CPU fallback with CPU-optimized vector indexes (FAISS CPU, BM25).


In [3]:
packages_to_check = [
    "datasets", "transformers", "sentence_transformers", "faiss", "rank_bm25",
    "pandas", "numpy", "pyarrow", "fastapi", "pydantic", "requests", "httpx"
]

print("=== DEPENDENCY CHECKS ===")
for pkg in packages_to_check:
    try:
        __import__(pkg)
        print(f"  [OK] {pkg}")
    except ImportError:
        print(f"  [MISSING] {pkg}")

=== DEPENDENCY CHECKS ===
  [OK] datasets
  [OK] transformers
  [MISSING] sentence_transformers
  [MISSING] faiss
  [MISSING] rank_bm25
  [OK] pandas
  [OK] numpy
  [OK] pyarrow
  [MISSING] fastapi
  [OK] pydantic
  [OK] requests
  [OK] httpx


## Environment Audit Summary & Architecture Guidance

1. **Execution Host**: Windows Platform
2. **Data Strategy**: `ai4bharat/MSMARCO-XI` full dataset is ~55.6 GB Parquet. We will work with language-specific subsets (e.g. Hindi `hi`, Bengali `bn`, Tamil `ta`, Telugu `te`) and structured evaluation samples to maintain high throughput and low memory footprint.
3. **Retrieval Architecture**: Dense embeddings (BGE-M3 / E5 Multilingual) + Sparse BM25 indexed locally using CPU-optimized FAISS and `rank_bm25` for sub-200ms candidate retrieval.
4. **Voice STT & Generation APIs**: Sarvam AI API for Indic Speech-to-Text & Gemini / Sarvam API for grounded generation to meet strict hackathon latency and quality targets.

In [4]:
from datasets import load_dataset
import pandas as pd
import json

# List of supported Indic language codes
INDIC_LANGUAGES = {
    'as': 'Assamese',
    'bn': 'Bengali',
    'gu': 'Gujarati',
    'hi': 'Hindi',
    'kn': 'Kannada',
    'ml': 'Malayalam',
    'mr': 'Marathi',
    'ne': 'Nepali',
    'or': 'Odia',
    'pa': 'Punjabi',
    'sa': 'Sanskrit',
    'ta': 'Tamil',
    'te': 'Telugu',
    'ur': 'Urdu'
}

print(f"Total Supported Languages: {len(INDIC_LANGUAGES)}")
for code, name in INDIC_LANGUAGES.items():
    print(f"  {code}: {name}")

Total Supported Languages: 14
  as: Assamese
  bn: Bengali
  gu: Gujarati
  hi: Hindi
  kn: Kannada
  ml: Malayalam
  mr: Marathi
  ne: Nepali
  or: Odia
  pa: Punjabi
  sa: Sanskrit
  ta: Tamil
  te: Telugu
  ur: Urdu


In [5]:
# Code template to inspect a language sample (e.g. Hindi 'hi')
sample_lang = 'hi'
print(f"Loading {INDIC_LANGUAGES[sample_lang]} ('{sample_lang}') dataset split...")

try:
    dataset = load_dataset("ai4bharat/MSMARCO-XI", sample_lang, split="validation")
    print(f"Validation samples count: {len(dataset)}")
    first_example = dataset[0]
    
    print("\n--- RECORD SCHEMA FIELDS ---")
    for k in first_example.keys():
        print(f" - {k}: {type(first_example[k])}")
        
    print("\n--- EXAMPLE SAMPLE ---")
    print(f"Query ID: {first_example['query_id']}")
    print(f"Query Type: {first_example['query_type']}")
    print(f"Target Lang: {first_example['target_lang']}")
    print(f"Indic Query: {first_example['query']}")
    print(f"English Query: {first_example['Eng_Query']}")
    print(f"Indic Answer: {first_example['Answer']}")
    print(f"English Answer: {first_example['Eng_Answer']}")
    
    passages = first_example['passages']
    print(f"Total Passages per Query: {len(passages['Translated_passages'])}")
    print(f"Selected Flags: {passages['is_selected']}")
    print(f"First Passage (Indic): {passages['Translated_passages'][0][:150]}...")
    print(f"First Passage (English): {passages['English_passages'][0][:150]}...")

except Exception as e:
    print(f"Error loading dataset: {e}")
    

Loading Hindi ('hi') dataset split...


Repo card metadata block was not found. Setting CardData to empty.


Error loading dataset: BuilderConfig 'hi' not found. Available: ['default']
